In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# AC power P(t) diagram for varying phase differences between V(t) and I(t)

class ACPower:

    def __init__(self, V_m, I_m, phi):
        """
        Input voltage and curernt amplitudes, along with the relative phase
        difference between voltage and current, phi.
        """
        
        self.V_m = V_m
        self.I_m = I_m
        self.phi = np.radians(phi)

        # assume 50 Hz signals
        self.omega = 2 * np.pi * 50

        # time (ms) to include 5 full 50 Hz waves
        self.t = np.linspace(0, 100, 300)

        # plot initialisation
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(2, 1, figsize=(10,8))

        # artists list
        self.waves = []

    def _format_coeff(self, coeff):
        """
        Convert '1' to '' and "-1" to '-', otherwise round to 5 s.f.
        """
        if coeff == 1:  return ''
        if coeff == -1: return '-'
        return f"{coeff:.5g}"   

    def _format_phi(self, phi):
        """
        Correctly handle sign of phase, and no phase, and round to 5 s.f.
        Phase of current is negative phase difference.
        """
        if self.phi < 0:  return f"+{-phi:.5g}"
        if self.phi > 0:  return f"{-phi:.5g}"
        return ''

    def _build_axes(self, animating=False):
        """
        Create static plot elements.
        """

        self.fig.subplots_adjust(
            left=0.1, right=0.9,
            bottom=0.08, top=0.9,
            wspace=0.2, hspace=0.2
        )

        self.fig.suptitle("AC Power", fontsize=25)

        # V and I share top axis, P on bottom
        self.ax_v, self.ax_p = self.ax
        self.ax_i = self.ax_v.twinx()

        # axis formatting
        self.ax_v.set_title(r"Voltage $V(t)$ and Current $I(t)$", fontsize=15)
        formatted_V_m = self._format_coeff(self.V_m)
        self.ax_v.set_ylabel(
            rf"$V(t) = {formatted_V_m} \sin (\omega t)$ (V)",
            fontsize=12, color="red"
        )
        self.ax_v.axhline(0, linewidth=2, color="white", alpha=0.5)
        self.ax_v.set_xlim(0, 100)
        vi_lim = 1.1 * max(self.V_m, self.I_m)
        self.ax_v.set_ylim(-vi_lim, vi_lim)
        self.ax_v.grid(True, which="major", alpha=0.2)
        self.ax_v.tick_params(axis='y', colors="red")

        formatted_I_m = self._format_coeff(self.I_m)
        if animating:
            formatted_phi = "+\phi(t)"
        else:
            formatted_phi = self._format_phi(np.degrees(self.phi))
        self.ax_i.set_ylabel(
            rf"$I(t) = {formatted_I_m} \sin(\omega t {formatted_phi})$ (A)", 
            fontsize=12, color="yellow"
        )
        self.ax_i.set_ylim(-vi_lim, vi_lim)
        self.ax_i.tick_params(axis='y', colors="yellow")

        self.ax_p.set_title(r"Power $P(t) = V(t) \cdot I(t)$", fontsize=15)
        self.ax_p.set_ylabel(r"$P(t)$ (W)", fontsize=12, color="blue")
        self.ax_p.set_xlabel(r"$Time (ms)$", fontsize=12)
        self.ax_p.set_xlim(0, 100)
        p_lim = 1.1 * self.V_m * self.I_m
        self.ax_p.set_ylim(-p_lim, p_lim)
        self.ax_p.axhline(0, linewidth=2, color="white", alpha=0.5)
        self.ax_p.grid(True, which="major", alpha=0.2)
        self.ax_p.tick_params(axis='y', colors="blue")

    def _init_artists(self):
        """
        Initialise dynamic plot elements (sinusoids) for each axis.
        """

        # voltage to remain reference sinusoid
        self.ax_v.plot(
            self.t, self.V_m * np.sin(self.omega * self.t*1e-3), 
            color="red", linewidth=2
        )

        # current and power to dynamically change with phase differnce
        self.waves.clear()
        zeros = np.zeros_like(self.t)
        self.waves.append(
            self.ax_i.plot(self.t, zeros, color="yellow", linewidth=2)[0]
        )
        self.waves.append(
            self.ax_p.plot(self.t, zeros, color="blue", linewidth=2)[0]
        )

    def _update(self, dphi):
        """
        For animation, step forward phase difference
        """

        # extract wave artists
        I, P = self.waves

        # update sinusoids
        I.set_ydata(
            self.I_m * np.sin(self.omega * self.t*1e-3 - dphi)
        )
        P.set_ydata(
            0.5 * self.V_m * self.I_m * (
                np.cos(dphi) - 
                np.cos(2 * self.omega * self.t*1e-3 - dphi)
            )
        )

    def plot(self):
        """
        Plot all current waves with current phase difference.
        """
        self._build_axes()
        self._init_artists()
        self._update(self.phi)
        self.fig.canvas.draw_idle()

    def animate(self, frames, save=False, title="AC-power"):
        """
        Animate plot through all phase differences
        """

        self._build_axes(animating=True)
        self._init_artists()

        # loop animation over 2 50 Hz cycles
        dphi = np.linspace(2*np.pi, 0, frames, endpoint=False)

        def update(frame):
            self._update(dphi[frame])
            return (self.waves)
                    
        anim = FuncAnimation(
            self.fig,
            update,
            frames=frames, 
            blit=True
        )

        # save at 5 fps
        if save:
            anim.save(f"{title}.gif", writer=PillowWriter(fps=frames/5))

        plt.close(self.fig)

        return HTML(anim.to_jshtml())